In [ ]:
# -------------------------------------------------------------
# Libraries and Imports
# -------------------------------------------------------------

# Paths and File Handling
import subprocess
from pathlib import Path
from jinja2 import Template
from typing import Optional
import json
import uuid
from dataclasses import dataclass, field, asdict
from typing import Optional
import gmsh
import pyarrow as pa
import pyarrow.parquet as pq
from uuid import uuid4

# Numerical and Visualization Libraries
import numpy as np
import math
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pyvista as pv
import h5py

# Neural Network and Machine Learning Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from scipy.ndimage import label
from scipy.ndimage import distance_transform_edt
from scipy.spatial import cKDTree
from scipy.interpolate import griddata
from scipy.stats import binned_statistic_2d
from concurrent.futures import ThreadPoolExecutor

# Additional utilities
import os
import time
import random
import copy
import warnings
warnings.filterwarnings("ignore", message="An output with one or more elements was resized")
from collections import defaultdict
from __future__ import annotations
from typing import Iterable, Sequence, Any, Callable

/home/austingardiner/miniforge/envs/phd/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# -------------------------------------------------------------
# Torch Device Configuration
# -------------------------------------------------------------
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon GPU")
else:
    device = torch.device("cpu")
    print("Using CPU")

In [ ]:
"""Gmsh mesh generation for the 2D cold-plate design domain.

The generator's design pixels are 33 1/3 um. CFD element sizes are separately
controlled and may be refined near fin walls. Coordinates and polygon vertices
are in metres. The domain is x=0..Lx, y=0..Ly, with inlet left, outflow right,
physical no-slip bottom, and horizontal symmetry at the top.
"""

DESIGN_PIXEL_M = 1.0 / 30_000.0
# Placeholder until the chip-to-chip spacing is measured. The geometry builder
# can use this value to form exclusion regions once the real gap is known.
CHIP_GAP_M = None


def snap_to_design_pixel(value: float, pixel_m: float = DESIGN_PIXEL_M) -> float:
    """Snap a coordinate or feature dimension to the manufacturing XY grid."""
    if pixel_m <= 0:
        raise ValueError("pixel_m must be positive")
    return round(value / pixel_m) * pixel_m


def _classify_boundary_curves(fluid_surfaces, Lx: float, Ly: float, tol: float = 1e-6):
    """Classify unique fluid-boundary curves by their entire bounding box."""
    groups = {"Inlet": [], "Outlet": [], "Bottom Wall": [], "Symmetry": [], "Fin Walls": []}
    seen = set()
    boundary = gmsh.model.getBoundary(
        [(2, s) for s in fluid_surfaces], combined=True, oriented=False
    )
    for dim, tag in boundary:
        if tag in seen:
            continue
        seen.add(tag)
        xmin, ymin, _, xmax, ymax, _ = gmsh.model.getBoundingBox(dim, tag)
        if abs(xmin) <= tol and abs(xmax) <= tol:
            groups["Inlet"].append(tag)
        elif abs(xmin - Lx) <= tol and abs(xmax - Lx) <= tol:
            groups["Outlet"].append(tag)
        elif abs(ymin) <= tol and abs(ymax) <= tol:
            groups["Bottom Wall"].append(tag)
        elif abs(ymin - Ly) <= tol and abs(ymax - Ly) <= tol:
            groups["Symmetry"].append(tag)
        else:
            groups["Fin Walls"].append(tag)
    return groups


def generate_mesh(
    Lx: float,
    Ly: float,
    nx: int,
    ny: int,
    fin_coords: Iterable[Sequence[Sequence[float]]],
    design_id: str,
    output_path: str,
    *,
    size_min: float | None = None,
    size_max: float | None = None,
    refine_dist: float | None = None,
    refine_outer_walls: bool = False,
    pixel_m: float = DESIGN_PIXEL_M,
    verbose: bool = False,
) -> str:
    """Create an all-quad 2D mesh, locally refined near fin and selected walls.

    ``size_min`` and ``size_max`` describe final post-subdivision element sizes.
    The structured design grid and the CFD mesh are intentionally distinct.
    Fins are polygons removed from the fluid rectangle; every polygon vertex
    must lie on the design pixel grid.
    """
    if Lx <= 0 or Ly <= 0 or nx < 1 or ny < 1:
        raise ValueError("Lx/Ly must be positive and nx/ny must be >= 1")
    dx, dy = Lx / nx, Ly / ny
    if size_min is None:
        size_min = min(dx, dy) / 2
    if size_max is None:
        size_max = max(dx, dy)
    if refine_dist is None:
        refine_dist = max(Lx, Ly) / 10
    if not (0 < size_min <= size_max) or refine_dist <= 0:
        raise ValueError("Require 0 < size_min <= size_max and refine_dist > 0")

    fin_coords = list(fin_coords)
    # Enforce pixel alignment to avoid silently changing the printable layout.
    for poly in fin_coords:
        for point in poly:
            for coordinate in point[:2]:
                if abs(coordinate - snap_to_design_pixel(coordinate, pixel_m)) > 1e-9:
                    raise ValueError(f"Fin coordinate {coordinate} is off the design pixel grid")

    gmsh.initialize()
    try:
        gmsh.option.setNumber("General.Terminal", 1 if verbose else 0)
        gmsh.model.add(f"cold_plate_{design_id}")

        rect = gmsh.model.occ.addRectangle(0, 0, 0, Lx, Ly)
        obstructions = []
        for polygon in fin_coords:
            points = [gmsh.model.occ.addPoint(x, y, 0) for x, y in polygon]
            lines = [gmsh.model.occ.addLine(points[i], points[(i + 1) % len(points)])
                     for i in range(len(points))]
            loop = gmsh.model.occ.addCurveLoop(lines)
            obstructions.append(gmsh.model.occ.addPlaneSurface([loop]))
        gmsh.model.occ.synchronize()

        if obstructions:
            cut, _ = gmsh.model.occ.cut([(2, rect)], [(2, s) for s in obstructions],
                                        removeObject=True, removeTool=True)
            gmsh.model.occ.synchronize()
            fluid_surfaces = [tag for dim, tag in cut if dim == 2]
        else:
            fluid_surfaces = [rect]
        if not fluid_surfaces:
            raise RuntimeError("Boolean subtraction produced no fluid surfaces")

        fluid_group = gmsh.model.addPhysicalGroup(2, fluid_surfaces)
        gmsh.model.setPhysicalName(2, fluid_group, "Fluid")
        groups = _classify_boundary_curves(fluid_surfaces, Lx, Ly)
        for name, tags in groups.items():
            if tags:
                group = gmsh.model.addPhysicalGroup(1, tags)
                gmsh.model.setPhysicalName(1, group, name)

        # Size is driven only by the background field to avoid point/curvature
        # sizing overriding wall refinement. Subdivision halves final sizes.
        subdivision_factor = 2.0
        h_min, h_max = size_min * subdivision_factor, size_max * subdivision_factor
        gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
        gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)
        gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
        gmsh.option.setNumber("Mesh.MeshSizeMin", h_min)
        gmsh.option.setNumber("Mesh.MeshSizeMax", h_max)

        refine_curves = list(groups["Fin Walls"])
        if refine_outer_walls:
            refine_curves.extend(groups["Bottom Wall"])
            refine_curves.extend(groups["Symmetry"])
        if refine_curves:
            distance = gmsh.model.mesh.field.add("Distance")
            gmsh.model.mesh.field.setNumbers(distance, "CurvesList", refine_curves)
            gmsh.model.mesh.field.setNumber(distance, "Sampling", 100)
            threshold = gmsh.model.mesh.field.add("Threshold")
            gmsh.model.mesh.field.setNumber(threshold, "InField", distance)
            gmsh.model.mesh.field.setNumber(threshold, "SizeMin", h_min)
            gmsh.model.mesh.field.setNumber(threshold, "SizeMax", h_max)
            gmsh.model.mesh.field.setNumber(threshold, "DistMin", 0)
            gmsh.model.mesh.field.setNumber(threshold, "DistMax", refine_dist)
            gmsh.model.mesh.field.setAsBackgroundMesh(threshold)

        gmsh.option.setNumber("Mesh.Algorithm", 8)
        gmsh.option.setNumber("Mesh.RecombinationAlgorithm", 3)
        gmsh.option.setNumber("Mesh.RecombineAll", 1)
        gmsh.option.setNumber("Mesh.SubdivisionAlgorithm", 1)
        gmsh.option.setNumber("Mesh.ElementOrder", 2)
        gmsh.option.setNumber("Mesh.SecondOrderLinear", 1)
        gmsh.model.mesh.generate(2)

        element_types, _, _ = gmsh.model.mesh.getElements(2)
        quad_types = {3, 10, 16}
        bad_types = [t for t in element_types if t not in quad_types]
        if bad_types:
            bad_names = [gmsh.model.mesh.getElementProperties(t)[0] for t in bad_types]
            raise RuntimeError(f"Mesh contains non-quad elements: {bad_names}")
        gmsh.write(output_path)
        return output_path
    finally:
        gmsh.finalize()


In [ ]:
"""Simple grid-based cold-plate geometry generation and design storage.

Occupancy convention: ``True`` is copper/solid and ``False`` is fluid.
Coordinates use x left-to-right and y bottom-to-top. The generator grid is the
33 1/3 um XY manufacturing grid; the CFD mesh is generated separately.
"""






MIN_FEATURE_MM = 0.100
MAX_FIN_THICKNESS_MM = 0.500
DEFAULT_ELEMENT_SIZE_UM = 100.0 / 3.0


def _jsonable(value):
    if isinstance(value, dict):
        return {str(k): _jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def _run_components(mask: np.ndarray):
    """Label 4-connected components using row runs; return component summaries.

    Each summary contains the component bounding box and whether it touches
    any domain edge, the inlet edge, and the outlet edge. Run-length labeling
    avoids a Python queue visit for every cell on the high-resolution grid.
    """
    ny, nx = mask.shape
    parent: list[int] = []
    boxes: list[list[int]] = []  # xmin, ymin, xmax, ymax
    run_nodes: list[tuple[int, int, int, int]] = []  # node, x0, x1-exclusive, y

    def find(a: int) -> int:
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def union(a: int, b: int) -> None:
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        parent[rb] = ra
        ba, bb = boxes[ra], boxes[rb]
        ba[0] = min(ba[0], bb[0]); ba[1] = min(ba[1], bb[1])
        ba[2] = max(ba[2], bb[2]); ba[3] = max(ba[3], bb[3])

    previous: list[tuple[int, int, int]] = []  # x0, x1-exclusive, node
    for y in range(ny):
        row = mask[y]
        changes = np.diff(np.concatenate(([False], row, [False])).astype(np.int8))
        starts = np.flatnonzero(changes == 1)
        stops = np.flatnonzero(changes == -1)
        current: list[tuple[int, int, int]] = []
        p = 0
        for x0, x1 in zip(starts.tolist(), stops.tolist()):
            node = len(parent)
            parent.append(node)
            boxes.append([x0, y, x1 - 1, y])
            current.append((x0, x1, node))
            while p < len(previous) and previous[p][1] <= x0:
                p += 1
            q = p
            while q < len(previous) and previous[q][0] < x1:
                if previous[q][1] > x0:
                    union(node, previous[q][2])
                q += 1
            run_nodes.append((node, x0, x1, y))
        previous = current

    summaries: dict[int, dict[str, Any]] = {}
    for node, x0, x1, y in run_nodes:
        root = find(node)
        entry = summaries.setdefault(root, {
            "bbox": boxes[root].copy(), "touches_boundary": False,
            "touches_inlet": False, "touches_outlet": False,
        })
        entry["bbox"] = boxes[root].copy()
        entry["touches_inlet"] |= x0 == 0
        entry["touches_outlet"] |= x1 == nx
        entry["touches_boundary"] |= y == 0 or y == ny - 1 or x0 == 0 or x1 == nx
    return list(summaries.values())


def _line_runs(line: np.ndarray):
    changes = np.diff(np.concatenate(([False], line, [False])).astype(np.int8))
    starts = np.flatnonzero(changes == 1)
    stops = np.flatnonzero(changes == -1)
    return zip(starts.tolist(), stops.tolist())


def _mask_to_polygons(solid: np.ndarray, dx_mm: float, dy_mm: float):
    """Convert a pixel solid mask into disjoint boundary polygons in mm."""
    ny, nx = solid.shape
    edges: set[tuple[tuple[int, int], tuple[int, int]]] = set()
    ys, xs = np.nonzero(solid)
    for y, x in zip(ys.tolist(), xs.tolist()):
        if y == 0 or not solid[y - 1, x]:
            edges.add(((x, y), (x + 1, y)))
        if x == nx - 1 or not solid[y, x + 1]:
            edges.add(((x + 1, y), (x + 1, y + 1)))
        if y == ny - 1 or not solid[y + 1, x]:
            edges.add(((x + 1, y + 1), (x, y + 1)))
        if x == 0 or not solid[y, x - 1]:
            edges.add(((x, y + 1), (x, y)))

    outgoing: dict[tuple[int, int], list[tuple[int, int]]] = defaultdict(list)
    for a, b in edges:
        outgoing[a].append(b)
    direction = {(1, 0): 0, (0, 1): 1, (-1, 0): 2, (0, -1): 3}
    polygons = []
    unused = set(edges)
    while unused:
        start_edge = next(iter(unused))
        start, current = start_edge
        loop = [start]
        previous = start
        unused.remove(start_edge)
        guard = 0
        while current != start:
            loop.append(current)
            incoming = direction[(current[0] - previous[0], current[1] - previous[1])]
            candidates = [end for end in outgoing[current] if (current, end) in unused]
            if not candidates:
                raise ValueError("Could not close a solid boundary loop")
            # At a grid vertex, prefer the leftmost turn to keep diagonal
            # contacts as separate loops rather than creating self-crossings.
            def turn_rank(end):
                d = direction[(end[0] - current[0], end[1] - current[1])]
                delta = (d - incoming) % 4
                return {1: 0, 0: 1, 3: 2, 2: 3}[delta]
            nxt = min(candidates, key=turn_rank)
            unused.remove((current, nxt))
            previous, current = current, nxt
            guard += 1
            if guard > len(edges) + 1:
                raise ValueError("Boundary tracing exceeded its safety limit")

        # Drop collinear grid vertices before creating OCC curves.
        compact = []
        for i, point in enumerate(loop):
            prev, nxt = loop[i - 1], loop[(i + 1) % len(loop)]
            if (point[0] - prev[0]) * (nxt[1] - point[1]) == \
               (point[1] - prev[1]) * (nxt[0] - point[0]):
                continue
            compact.append(point)
        if len(compact) >= 3:
            polygons.append([(x * dx_mm, y * dy_mm) for x, y in compact])
    return polygons

In [ ]:
@dataclass
class HeatExchanger:
    """One design: geometry, metadata, generated mesh path, and result fields."""

    occupancy_grid: np.ndarray
    fin_plate_x_mm: float
    fin_plate_y_mm: float
    element_size_mm: float = DEFAULT_ELEMENT_SIZE_UM / 1000.0
    design_id: str | None = None
    base_dir: str | Path = "."
    results: dict[str, Any] | None = None
    mesh_path: str | None = None
    metadata: dict[str, Any] | None = None

    def __post_init__(self):
        self.occupancy_grid = np.asarray(self.occupancy_grid, dtype=bool)
        if self.occupancy_grid.ndim != 2:
            raise ValueError("occupancy_grid must be a 2D array")
        if self.element_size_mm <= 0:
            raise ValueError("element_size_mm must be positive")
        if self.design_id is None:
            self.design_id = uuid4().hex[:12]
        self.base_dir = Path(self.base_dir)
        self.results = dict(self.results or {})
        self.metadata = dict(self.metadata or {})

    @property
    def grid_ny(self) -> int:
        return int(self.occupancy_grid.shape[0])

    @property
    def grid_nx(self) -> int:
        return int(self.occupancy_grid.shape[1])

    @property
    def actual_dimensions_mm(self) -> tuple[float, float]:
        return self.grid_nx * self.element_size_mm, self.grid_ny * self.element_size_mm

    def generate_mesh(self, output_path: str | Path | None = None, **mesh_options) -> str:
        """Generate a MOOSE-compatible Gmsh mesh from the stored occupancy grid."""
        if output_path is None:
            mesh_dir = self.base_dir / "meshes"
            mesh_dir.mkdir(parents=True, exist_ok=True)
            output_path = mesh_dir / f"{self.design_id}.msh"
        else:
            output_path = Path(output_path)
            output_path.parent.mkdir(parents=True, exist_ok=True)
        mesh_options.setdefault("pixel_m", self.element_size_mm / 1000.0)
        dx = dy = self.element_size_mm
        fin_polygons = _mask_to_polygons(self.occupancy_grid, dx, dy)
        Lx, Ly = self.actual_dimensions_mm
        result = generate_mesh(
            Lx=Lx / 1000.0, Ly=Ly / 1000.0,
            nx=self.grid_nx, ny=self.grid_ny,
            fin_coords=[[(x / 1000.0, y / 1000.0) for x, y in p] for p in fin_polygons],
            design_id=self.design_id, output_path=str(output_path),
            **mesh_options,
        )
        self.mesh_path = str(result)
        return self.mesh_path

    def read_moose_results(self, result_path: str | Path | None = None):
        """Stub: Exodus result reading will be implemented with the MOOSE setup."""
        pass

    def save(self, path: str | Path | None = None) -> str:
        """Save geometry, metadata, mesh reference, and results in one NPZ file."""
        if path is None:
            path = self.base_dir / f"{self.design_id}.npz"
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        meta = {
            "design_id": self.design_id,
            "fin_plate_x_mm": float(self.fin_plate_x_mm),
            "fin_plate_y_mm": float(self.fin_plate_y_mm),
            "element_size_mm": float(self.element_size_mm),
            "mesh_path": self.mesh_path,
            "metadata": _jsonable(self.metadata),
            "result_items": [],
        }
        arrays: dict[str, np.ndarray] = {"occupancy_grid": self.occupancy_grid.astype(np.uint8)}
        for i, (key, value) in enumerate(self.results.items()):
            if isinstance(value, (np.ndarray, list, tuple)):
                array_key = f"result_{i}"
                arrays[array_key] = np.asarray(value)
                meta["result_items"].append({"name": key, "array": array_key})
            else:
                try:
                    value_json = _jsonable(value)
                    json.dumps(value_json)
                except TypeError as exc:
                    raise TypeError(f"Result {key!r} is not serializable") from exc
                meta["result_items"].append({"name": key, "value": value_json})
        arrays["metadata_json"] = np.asarray(json.dumps(meta))
        with path.open("wb") as stream:
            np.savez_compressed(stream, **arrays)
        return str(path)

    @classmethod
    def read(cls, path: str | Path) -> "HeatExchanger":
        """Reload one saved design without requiring pickle-enabled archives."""
        path = Path(path)
        with np.load(path, allow_pickle=False) as saved:
            meta = json.loads(str(saved["metadata_json"].item()))
            results = {}
            for item in meta.get("result_items", []):
                if "array" in item:
                    results[item["name"]] = saved[item["array"]].copy()
                else:
                    results[item["name"]] = item.get("value")
            return cls(
                occupancy_grid=saved["occupancy_grid"].astype(bool),
                fin_plate_x_mm=meta["fin_plate_x_mm"],
                fin_plate_y_mm=meta["fin_plate_y_mm"],
                element_size_mm=meta["element_size_mm"],
                design_id=meta["design_id"],
                base_dir=path.parent,
                results=results,
                mesh_path=meta.get("mesh_path"),
                metadata=meta.get("metadata", {}),
            )

In [ ]:
class Generator:
    """Generate and validate grid-aligned cold-plate designs.

    ``neural_model`` is optional. For ``method='neural'`` it must accept a
    latent tensor of shape (1, latent_dim) and return occupancy probabilities
    shaped (ny, nx), (1, ny, nx), or (1, 1, ny, nx). This keeps the generator
    independent of a particular neural architecture or training framework.
    """

    def __init__(
        self,
        fin_plate_x_mm: float,
        fin_plate_y_mm: float,
        element_size_um: float = DEFAULT_ELEMENT_SIZE_UM,
        *,
        neural_model: Callable | Any | None = None,
        latent_dim: int = 32,
        seed: int | None = None,
        base_dir: str | Path = ".",
        max_attempts: int = 100,
    ):
        if fin_plate_x_mm <= 0 or fin_plate_y_mm <= 0 or element_size_um <= 0:
            raise ValueError("Plate dimensions and element size must be positive")
        if latent_dim < 1 or max_attempts < 1:
            raise ValueError("latent_dim and max_attempts must be positive")
        self.fin_plate_x_mm = float(fin_plate_x_mm)
        self.fin_plate_y_mm = float(fin_plate_y_mm)
        self.element_size_mm = float(element_size_um) / 1000.0
        self.element_size_um = float(element_size_um)
        if math.ceil(MIN_FEATURE_MM / self.element_size_mm - 1e-10) > \
           math.floor(MAX_FIN_THICKNESS_MM / self.element_size_mm + 1e-10):
            raise ValueError("Element size is incompatible with the 0.100-0.500 mm feature limits")
        # Keep the requested plate envelope as an upper bound. If a dimension
        # is not an integer number of pixels, the final row/column is omitted.
        self.grid_nx = int(math.floor(self.fin_plate_x_mm / self.element_size_mm + 1e-10))
        self.grid_ny = int(math.floor(self.fin_plate_y_mm / self.element_size_mm + 1e-10))
        if self.grid_nx < 3 or self.grid_ny < 3:
            raise ValueError("Plate dimensions must contain at least a 3 x 3 pixel grid")
        self.neural_model = neural_model
        self.latent_dim = int(latent_dim)
        self.rng = np.random.default_rng(seed)
        self.base_dir = Path(base_dir)
        self.max_attempts = int(max_attempts)
        self.last_validation_report: dict[str, Any] | None = None

    @property
    def actual_dimensions_mm(self) -> tuple[float, float]:
        return self.grid_nx * self.element_size_mm, self.grid_ny * self.element_size_mm

    def validate(self, occupancy_grid: np.ndarray) -> tuple[bool, dict[str, Any]]:
        """Check flow connectivity, enclosed pockets, and XY feature limits.

        Feature checks use 4-connected pixel components and their bounding-box
        thickness as a first-pass manufacturability screen. A final CAD/DfAM
        review is still needed for true 3D overhang angle and local geometry.
        """
        solid = np.asarray(occupancy_grid, dtype=bool)
        errors: list[str] = []
        if solid.shape != (self.grid_ny, self.grid_nx):
            return False, {"errors": [f"Expected grid {(self.grid_ny, self.grid_nx)}, got {solid.shape}"]}
        ny, nx = solid.shape
        min_cells = int(math.ceil(MIN_FEATURE_MM / self.element_size_mm - 1e-10))
        max_cells = int(math.floor(MAX_FIN_THICKNESS_MM / self.element_size_mm + 1e-10))

        # Keep the flow opening connected to explicit inlet/outlet sides.
        # Solid on an external edge would also make boundary classification
        # and the left/right boundary conditions ambiguous.
        if solid[:, 0].any() or solid[:, -1].any():
            errors.append("Solid cells touch the inlet or outlet edge")
        fluid_components = _run_components(~solid)
        if any(not c["touches_boundary"] for c in fluid_components):
            errors.append("Enclosed fluid pocket detected")
        if not any(c["touches_inlet"] and c["touches_outlet"] for c in fluid_components):
            errors.append("No continuous inlet-to-outlet fluid path")

        solid_components = _run_components(solid)
        for component in solid_components:
            xmin, ymin, xmax, ymax = component["bbox"]
            thickness = min(xmax - xmin + 1, ymax - ymin + 1)
            if thickness < min_cells:
                errors.append(f"Solid feature below minimum {MIN_FEATURE_MM:.3f} mm")
                break
            if thickness > max_cells:
                errors.append(f"Fin thickness exceeds maximum {MAX_FIN_THICKNESS_MM:.3f} mm")
                break

        # Reject narrow fluid slits trapped between fins in either grid axis.
        for line in list(~solid) + list((~solid).T):
            for start, stop in _line_runs(line):
                bounded = start > 0 and stop < len(line)
                if bounded and stop - start < min_cells:
                    errors.append("Fluid clearance below the minimum feature size")
                    break
            if errors and errors[-1] == "Fluid clearance below the minimum feature size":
                break

        return not errors, {
            "valid": not errors,
            "errors": errors,
            "min_feature_cells": min_cells,
            "max_fin_thickness_cells": max_cells,
            "solid_fraction": float(solid.mean()),
            "fluid_components": len(fluid_components),
            "solid_components": len(solid_components),
            "xy_only": True,
            "limitations": ["3D overhang angle is not checked by this 2D generator"],
        }

    def _random_grid(self) -> np.ndarray:
        """Create separated, grid-aligned streamwise fin segments."""
        ny, nx = self.grid_ny, self.grid_nx
        min_cells = int(math.ceil(MIN_FEATURE_MM / self.element_size_mm - 1e-10))
        max_cells = int(math.floor(MAX_FIN_THICKNESS_MM / self.element_size_mm + 1e-10))
        if min(ny, nx) < 2 * min_cells + 3:
            raise ValueError("Random fin generation needs a larger grid for clearances")
        solid = np.zeros((ny, nx), dtype=bool)
        clearance = min_cells
        occupied = np.zeros_like(solid)
        # Reserve an open strip so the random layout always has a through-flow
        # route; keep it away from both solid edges by a manufacturable gap.
        route_width = min_cells
        route_y0 = max(min_cells, ny // 2 - route_width // 2)
        route_y1 = route_y0 + route_width
        target = min(500, max(12, int((nx * ny * 0.06) / (60 * 8))))
        placed = 0
        attempts = 0
        while placed < target and attempts < target * 80:
            attempts += 1
            horizontal = bool(self.rng.random() < 0.8)
            thickness = int(self.rng.integers(min_cells, max_cells + 1))
            long_max = min(nx - 2 * clearance, max(6 * min_cells, 120))
            if long_max < 2 * min_cells:
                break
            length = int(self.rng.integers(2 * min_cells, long_max + 1))
            if horizontal:
                width_x, width_y = length, thickness
            else:
                width_x, width_y = thickness, length
            if width_x > nx - 2 * clearance or width_y > ny - 2 * clearance:
                continue
            x0 = int(self.rng.integers(clearance, nx - clearance - width_x + 1))
            y0 = int(self.rng.integers(clearance, ny - clearance - width_y + 1))
            y1 = y0 + width_y
            if y0 < route_y1 + clearance and y1 > route_y0 - clearance:
                continue
            ex0, ex1 = max(0, x0 - clearance), min(nx, x0 + width_x + clearance)
            ey0, ey1 = max(0, y0 - clearance), min(ny, y1 + clearance)
            if occupied[ey0:ey1, ex0:ex1].any():
                continue
            solid[y0:y1, x0:x0 + width_x] = True
            occupied[ey0:ey1, ex0:ex1] = True
            placed += 1
        return solid

    def _neural_grid(self) -> np.ndarray:
        if self.neural_model is None:
            raise RuntimeError("method='neural' requires a trained neural_model at construction")
        latent = self.rng.standard_normal((1, self.latent_dim)).astype(np.float32)
        if hasattr(self.neural_model, "generate_grid"):
            raw = self.neural_model.generate_grid(self.grid_ny, self.grid_nx, latent[0])
            if hasattr(raw, "detach"):
                raw = raw.detach().cpu().numpy()
        else:
            try:
                import torch
            except ImportError as exc:
                raise RuntimeError("PyTorch is required for a callable neural_model") from exc
            model = self.neural_model
            was_training = getattr(model, "training", None)
            if hasattr(model, "eval"):
                model.eval()
            with torch.no_grad():
                raw = model(torch.from_numpy(latent))
            if was_training and hasattr(model, "train"):
                model.train()
            if hasattr(raw, "detach"):
                raw = raw.detach().cpu().numpy()
        grid = np.asarray(raw, dtype=np.float32).squeeze()
        if grid.shape != (self.grid_ny, self.grid_nx):
            raise ValueError(
                f"neural_model returned {grid.shape}; expected {(self.grid_ny, self.grid_nx)}"
            )
        if not np.isfinite(grid).all():
            raise ValueError("neural_model output contains NaN or infinite values")
        return grid >= 0.5  # model contract: occupancy probabilities, solid-high

    def generate(self, method: str = "random") -> HeatExchanger:
        """Generate one valid HeatExchanger using ``random`` or ``neural``."""
        method = method.lower()
        if method not in {"random", "neural"}:
            raise ValueError("method must be 'random' or 'neural'")
        for _ in range(self.max_attempts):
            grid = self._random_grid() if method == "random" else self._neural_grid()
            valid, report = self.validate(grid)
            self.last_validation_report = report
            if valid:
                return HeatExchanger(
                    occupancy_grid=grid,
                    fin_plate_x_mm=self.fin_plate_x_mm,
                    fin_plate_y_mm=self.fin_plate_y_mm,
                    element_size_mm=self.element_size_mm,
                    base_dir=self.base_dir,
                    metadata={"generation_method": method, "validation": report},
                )
        detail = self.last_validation_report or {}
        raise RuntimeError(f"Could not generate a valid {method} design in {self.max_attempts} attempts: {detail}")

    def generate_batch(self, n_designs: int, method: str = "random") -> list[HeatExchanger]:
        """Generate ``n_designs`` valid HeatExchanger objects."""
        if n_designs < 0:
            raise ValueError("n_designs must be nonnegative")
        return [self.generate(method=method) for _ in range(n_designs)]